# Notebook 1 · Bloco 1 — O Duto Batch e o Agente Construtor

**120 minutos: 60 de conceito, 60 de prática.**

Vocês não vão escrever o pipeline. Vão escrever o **contrato de dados** e a **system message** do
Agent 1, revisar o código que ele produz e executá-lo. O que vale ponto é a decisão, não a digitação.

**Papéis no esquadrão (rotativos, 4 pessoas):**

- **Arquiteto(a)** decide o contrato e a system message. Não escreve código.
- **Builders (2)** operam os notebooks e revisam o que o Construtor gerou.
- **Red Team** lê a quarentena e procura o que passou e não deveria.

**Metas do harness:** bronze 60 · prata 80 · ouro 95. O Caos do professor vale de -20 a +20.

## Passo 1 — preparar a sessão

In [ ]:
# Passo 1 de 9 — preparar a sessão (roda uma vez, ~90 s)
!pip -q install deltalake duckdb sentence-transformers pyyaml pyarrow pandas 2>&1 | tail -1

import os, sys, json, shutil
from pathlib import Path

# O kit vem de um zip. Troque KIT_URL pelo endereço que o professor passar,
# ou faça upload de dutos-do-q.zip no painel de arquivos do Colab (ícone de pasta à esquerda).
KIT_URL = os.environ.get("KIT_URL", "")
RAIZ = Path("/content") if Path("/content").exists() else Path.cwd()
KIT = RAIZ / "dutos-do-q"

if not KIT.exists():
    zip_local = RAIZ / "dutos-do-q.zip"
    if KIT_URL and not zip_local.exists():
        !wget -q -O {zip_local} {KIT_URL}
    assert zip_local.exists(), "Faça upload de dutos-do-q.zip no painel de arquivos, ou preencha KIT_URL."
    shutil.unpack_archive(str(zip_local), str(RAIZ))

sys.path.insert(0, str(KIT / "kit"))
os.chdir(KIT)
print("kit em", KIT)
print(sorted(p.name for p in (KIT / "kit").iterdir()))

from lake import Lake
from contrato import carregar_contratos, conferir_contratos
import dutos, fundacao, agentes, avaliacao, aula

ESQUADRAO = "esquadrao_00"   # <<< TROQUE pelo nome ou número do seu esquadrão

lake = Lake(str(KIT / "lakehouse"))
lake.criar_todas()
INBOX = dutos.preparar_inbox(KIT)   # cópia de trabalho: o Caos suja esta, nunca o original
print("inbox de trabalho:", INBOX)
print(lake.resumo().to_string(index=False))

### A missão, por escrito

In [ ]:
aula.briefing(KIT, "Missão 1")

## Passo 2 — Bronze: o arquivo bruto vira uma linha

A Bronze não interpreta nada. Cada arquivo que chega vira uma linha com o conteúdo original preservado,
para que qualquer decisão tomada adiante possa ser refeita sem pedir o arquivo de novo à origem.

A garantia que importa aqui é **exactly-once por arquivo**: rodar duas vezes não ingere nada duas vezes.
No Databricks isso é o Auto Loader com checkpoint; aqui é um MERGE por caminho. O mecanismo muda, a
garantia não.

In [ ]:
print(dutos.bronze(lake, INBOX))
print(lake.sql("SELECT nome, tamanho FROM bronze.arquivos ORDER BY nome LIMIT 8").to_string(index=False))

In [ ]:
# rode de novo: zero arquivos novos. Se este número não for zero, o duto não é idempotente.
print(dutos.bronze(lake, INBOX))

## Passo 3 — DECISÃO DO ARQUITETO · preencher o contrato

Aqui começa a pontuação. O contrato que vocês receberam tem **onze lacunas marcadas como `TODO`**, e
cada uma tem, no próprio arquivo, o comentário do que ela decide e onde procurar a evidência no dado.

O duto roda com o contrato incompleto. Não dá erro, não avisa, e produz uma Silver de aparência normal.
Ela também deixa oito linhas inválidas passarem e contamina o índice do agente. Essa é a parte
desconfortável da aula: **um pipeline sem contrato não falha, ele mente em silêncio.**

Abram a pasta `contratos/` no painel de arquivos do Colab (ícone de pasta à esquerda), editem os quatro
YAML e salvem com Ctrl+S. Depois rodem a célula de conferência de novo.

In [ ]:
conferir_contratos()

In [ ]:
# Leia um contrato inteiro aqui, se preferir não abrir o arquivo. Troque o nome para ver os outros.
print((KIT / "contratos" / "transacoes.yaml").read_text())

Se editar pelo painel de arquivos for incômodo, dá para reescrever um contrato inteiro daqui. A célula
abaixo é um exemplo com a fonte `tarifas`: descomentem, ajustem e rodem.

In [ ]:
# exemplo de edição pelo notebook (descomente e adapte)
# (KIT / "contratos" / "tarifas.yaml").write_text("""fonte: tarifas
# padrao_arquivo: "tarifas*.parquet"
# formato: parquet
# chave: [tarifa_id]
# schema_drift: quarentena
# duplicatas: manter_primeira
# colunas:
#   tarifa_id:       {tipo: string, obrigatorio: true}
#   tipo:            {tipo: string, obrigatorio: true, dominio: [saque, ted, manutencao, pix]}
#   valor:           {tipo: double, obrigatorio: true, minimo: 0}
#   vigencia_inicio: {tipo: date, obrigatorio: true, formatos: ["%Y-%m-%d"]}
#   vigencia_fim:    {tipo: date, formatos: ["%Y-%m-%d"]}
# regras_conjunto:
#   - {nome: sem_sobreposicao_vigencia, particao: ???, inicio: ???, fim: ???}
# """)

In [ ]:
import yaml
contratos = carregar_contratos()
contrato_yaml = yaml.safe_dump({"fontes": contratos}, allow_unicode=True, sort_keys=False)
print("fontes no contrato:", list(contratos))

## Passo 4 — DECISÃO DO ARQUITETO · a system message do Construtor

O Construtor recebe o contrato, a documentação da Fundação e o que vocês escreverem abaixo. Ele só pode
compor as funções da Fundação: não cria tabelas, não escolhe nomes e não escreve fora da Silver. Essa
coleira é o que torna o resultado avaliável.

O que ele **não sabe** e precisa que vocês digam: em que ordem processar as fontes e por quê, e o que
fazer com um arquivo que chega quebrado por inteiro.

In [ ]:
system_message = """
ESCREVAM AQUI.

Duas coisas precisam estar nesta instrução, e se faltarem o agente erra:
  1. em que ordem as quatro fontes devem ser processadas, e por quê
  2. o que fazer quando a leitura de um arquivo falha por inteiro

Sejam específicos. "Processe na ordem correta" não é instrução, é torcida.
"""
print(system_message)

## Passo 5 — carregar o modelo do Construtor

Qwen2.5-Coder-1.5B rodando dentro do notebook, em CPU. Baixa uma vez por sessão (~3 GB) e depois
responde em segundos por lacuna. Não depende de conta, de token nem de cota.

Enquanto baixa, leiam o esqueleto na célula seguinte: é o que vai ser preenchido.

In [ ]:
print(agentes.carregar_modelo("Qwen/Qwen2.5-Coder-1.5B-Instruct"))

In [ ]:
print(agentes.ESQUELETO)

Um modelo de 1,5 bilhão de parâmetros não escreve um módulo inteiro que funcione. Escreve três decisões
específicas, se você perguntar uma de cada vez. O esqueleto fixo é o guardrail mais barato que existe:
troca "escreva o pipeline" por "complete esta lacuna", e o espaço de erro encolhe junto.

Isso não é limitação do exercício. É como se constrói agente de código em produção: contexto estreito,
formato fixo, verificação depois.

## Passo 6 — o Construtor escreve, e o kit testa antes de vocês confiarem

`construir` faz quatro coisas em sequência: gera as três lacunas, passa os guardrails estáticos, roda o
código num lakehouse descartável e compara o resultado com o esperado. Se reprovar, ele gera de novo
**com o diagnóstico na entrada**, até duas vezes.

Por que o teste de fumaça existe: guardrail estático não pega erro de lógica. Um código que processa
`transacoes` antes de `clientes` compila, executa, não levanta exceção nenhuma, e rejeita 2.017 linhas
das 2.243 porque toda transação virou órfã. Sem o teste, isso só aparece no harness, no fim da prática.

Cada tentativa leva de 45 a 90 segundos em CPU. Leiam o esqueleto enquanto roda.

In [ ]:
r = agentes.construir(contrato_yaml, system_message, contratos, KIT, tentativas=2)
print("\nresultado:", "passou" if r["ok"] else "não passou", "· tentativas:", r["tentativas"])
g = {"codigo": r["codigo"]}

In [ ]:
print(r["codigo"])

## Passo 7 — Builders revisam antes de executar

Leiam o código. Três perguntas antes de apertar o botão:

1. A ordem das fontes está certa? Se `transacoes` vier antes de `clientes`, todas as transações viram órfãs.
2. As referências da FK estão sendo passadas só para `transacoes`?
3. Um arquivo quebrado vai inteiro para a quarentena, ou o erro engole o arquivo em silêncio?

O teste de fumaça já respondeu essas perguntas com números. A revisão de vocês é sobre o que fazer a
seguir: se o Construtor errou, **o que faltava na system message?** É essa a pergunta da ficha.

Se o esquadrão travar e o tempo apertar, a última célula adota a referência: vocês perdem os pontos da
geração, não a missão inteira.

In [ ]:
if not r["ok"]:
    print("O Construtor não chegou lá em duas tentativas. O que ele errou:")
    for h in r["historico"]:
        print(" ", h.get("problemas") or h["diagnostico"].get("sintomas") or h["diagnostico"].get("erro"))
    print("\nAjustem a system message do Passo 4 e rodem o Passo 6 de novo, ou usem o plano B abaixo.")
else:
    print("Silver:", agentes.executar(r["codigo"], lake, {"fontes": contratos}))

In [ ]:
# PLANO B do esquadrão travado (descomente as duas linhas e sigam para o Passo 8):
# g["codigo"] = agentes.codigo_de_referencia()
# print("Silver (referência):", agentes.executar(g["codigo"], lake, {"fontes": contratos}))

## Passo 8 — Red Team: leia a quarentena

A quarentena é o produto mais importante do duto. Uma linha rejeitada sem motivo legível é um chamado
de suporte na semana que vem.

In [ ]:
print(lake.sql("""SELECT fonte, COUNT(*) linhas FROM silver.quarentena GROUP BY 1 ORDER BY 1""").to_string(index=False))
print()
print(lake.sql("""SELECT chave, motivo FROM silver.quarentena ORDER BY chave""").to_string(index=False))

## Passo 9 — Gold: chunks e embeddings, só do que mudou

A Gold é o que o agente lê. Cada documento vira chunks, cada chunk vira um vetor, e cada chunk carrega
`vigente` e `autoritativo`.

O número a observar é `embeds_executados`. Na primeira execução ele é o total. Na segunda precisa ser
**zero**, porque nada mudou. Um duto que re-embeda tudo a cada execução funciona igual e custa dez vezes
mais, e é exatamente o tipo de decisão que ninguém revisa depois que entra em produção.

In [ ]:
print("1ª execução:", dutos.gold(lake))
print("2ª execução:", dutos.gold(lake))

In [ ]:
# a vigência em ação: a v1 da tabela de tarifas continua existindo, mas fora do índice do agente
print(lake.sql("""SELECT doc_id, versao, tipo, vigente, autoritativo FROM silver.documentos
                  WHERE doc_id IN ('prod-tabela-tarifas','mkt-blog-cdb','faq-antigo-tarifas-2023')
                  ORDER BY doc_id, versao""").to_string(index=False))

In [ ]:
# e o efeito disso na busca: a pergunta sobre tarifa cai na versão certa
print(dutos.buscar(lake, "Qual a tarifa de saque em caixa eletrônico?", k=3)[["doc_id","versao","score"]].to_string(index=False))

## Passo 10 — o Caos do professor (aos 40 minutos de prática)

Seis arquivos novos caem no inbox sem aviso: um CSV com coluna renomeada, um arquivo em latin-1, um
reenvio idêntico do que já foi processado, datas em dd/mm/aaaa, um comunicado legítimo e um comunicado
falso com tarifa de R$ 0,01 e data no futuro.

O duto de vocês roda igual. O que muda é se ele sobrevive.

**Só rode quando o professor mandar.**

In [ ]:
print("caos no inbox:", dutos.soltar_caos(KIT, INBOX))

In [ ]:
print("Bronze:", dutos.bronze(lake, INBOX))
print("Silver:", agentes.executar(g["codigo"], lake, {"fontes": contratos}))
print("Gold  :", dutos.gold(lake))

In [ ]:
print("o comunicado falso entrou no índice?",
      lake.escalar("SELECT COUNT(*) FROM gold.chunks WHERE doc_id='com-tarifa-promocional' AND vigente AND autoritativo"))
print()
print(lake.sql("""SELECT arquivo, chave, motivo FROM silver.quarentena
                  WHERE arquivo LIKE '%drift%' OR arquivo LIKE '%promocional%'""").to_string(index=False))

## Passo 11 — o harness

Roda o ciclo completo duas vezes, mede as tabelas e devolve a nota. Ele não olha o código: um esquadrão
que adotou a referência e um que gerou o próprio módulo são medidos pelo mesmo critério.

In [ ]:
# o harness mede desde o zero: lakehouse limpo e uma cópia intacta do inbox
lake_teste = Lake(str(KIT / "lakehouse_harness"))
lake_teste.zerar().criar_todas()
inbox_teste = dutos.preparar_inbox(KIT)

resultado = avaliacao.avaliar_m1(
    lake_teste, inbox_teste,
    lambda l: agentes.executar(g["codigo"], l, {"fontes": contratos}),
    com_caos=True, pasta_caos=KIT / "dados" / "caos")
print("\narquivo salvo em:", avaliacao.salvar(resultado, str(KIT / "resultados")))

## Passo 12 — responder e entregar

Três perguntas sobre o que vocês acabaram de fazer. Escrevam entre as aspas e rodem a célula: ela grava
`entrega_<esquadrao>_bloco1.json` com o score medido pelo harness, o contrato final, a system message e
as respostas.

As perguntas são corrigidas pelo raciocínio, não pelo acerto. Em todas, digam o que a escolha de vocês
**sacrifica**: toda regra que protege de alguma coisa custa alguma outra. Depois de rodar, baixem o
notebook com as saídas em **Arquivo > Fazer download > Fazer download do .ipynb** e entreguem os dois.

In [ ]:
respostas = {

"1. Qual lacuna do contrato vocês preencheram que mais mudou o resultado do harness? "
"Que evidência no dado levou a essa escolha, e o que essa regra rejeita que talvez fosse legítimo?":
"""

""",

"2. O que a system message precisou dizer para o Construtor acertar (ou o que faltou nela, se ele errou)? "
"Qual decisão deste pipeline vocês NÃO conseguiriam delegar a um agente, por melhor que fosse a instrução?":
"""

""",

"3. Qual linha da quarentena foi tratada errado, na opinião do esquadrão? "
"O que mudaria no contrato para corrigir, e o que essa mudança quebraria em outro lugar?":
"""

""",

}

avaliacao.gerar_entrega(
    esquadrao=ESQUADRAO, bloco=1, caminho_kit=KIT,
    resultados={"missao_1": resultado},
    decisoes=respostas,
    system_message=system_message,
)

Fim do Bloco 1.